# Satellite Image Land Cover Classification on EuroSAT

**COMP9444 Neural Networks and Deep Learning — Team Cake Project 54**

---

## 1. Introduction, motivation and problem statement

### Problem statement

Given a 64 x 64 pixel Sentinel-2 RGB satellite image patch, assign it to exactly one of ten
land use / land cover classes. This is a single-label ten-way image classification problem
over 27,000 georeferenced patches with mild class imbalance (2,000-3,000 images per class).

### Motivation

Land cover maps underpin environmental monitoring, urban planning, agricultural analysis and
disaster management. Producing them manually is slow and labour-intensive, and human
interpretation is inconsistent across analysts, seasons, atmospheric conditions and sensor
resolutions. The Sentinel-2 constellation revisits almost the entire land surface every five
days and its imagery is openly licensed, so an accurate automatic classifier converts a free,
continuous data stream into maps that can be kept current at continental scale. Helber et al.
demonstrate exactly this downstream value, using a EuroSAT-trained classifier to detect
deforestation, urban expansion and industrial demolition from image pairs taken years apart.

### Models and comparison

We fine-tune three ImageNet-pretrained architectures on EuroSAT under a fixed protocol. The
comparison covers accuracy, per-class behaviour, parameter count and training cost:

| Model | Family | Why included |
|---|---|---|
| ResNet50 | Classic deep residual CNN | The architecture used by the dataset authors; our reference point against published results |
| EfficientNet-B0 | Compound-scaled efficient CNN | Tests whether ~6x fewer parameters costs accuracy |
| DeiT-Tiny/16 | Vision Transformer (distillation-trained) | Tests whether attention beats convolution at a matched parameter budget |

EfficientNet-B0 (4.0M) and DeiT-Tiny (5.5M) are close in size. Their similar capacity provides
a roughly parameter-controlled comparison between convolution and attention.

All three models share one seed-42 stratified 70/15/15 split, one preprocessing pipeline, one
optimiser configuration and one evaluation harness. Each predefined training run evaluates the
test set once, after validation-based checkpoint selection. Test results do not influence
training or model selection.

### Notebook structure

| Section | Content |
|---|---|
| 2-4 | Data source, experimental setup, leakage and integrity audit |
| 5 | Exploratory data analysis |
| 6 | Models, training protocol and attribution |
| 7-9 | Per-model training and evaluation |
| 10-11 | Cross-model comparison and benchmarking against published results |
| 12-13 | Discussion and conclusion |

## 2. Data source

| Property | Value |
|---|---|
| Dataset | EuroSAT, RGB version |
| Reference | P. Helber, B. Bischke, A. Dengel, D. Borth, "EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification", *IEEE JSTARS* 12(7):2217-2226, 2019 |
| Download | <https://zenodo.org/records/7711810> |
| Sensor | Sentinel-2A Multispectral Imager, ESA Copernicus programme |
| Images | 27,000 labelled, georeferenced patches |
| Classes | 10 (listed below) |
| Patch size | 64 x 64 pixels |
| Ground sampling distance | 10 m/pixel, so one patch covers 640 m x 640 m |
| Bands used here | B04 / B03 / B02 (red, green, blue). The full dataset also ships all 13 Sentinel-2 bands |
| Geographic coverage | 34 European countries |
| Licence | Free and open for commercial and non-commercial use |

**Class indices (fixed throughout, do not reorder):**

`0 AnnualCrop` · `1 Forest` · `2 HerbaceousVegetation` · `3 Highway` · `4 Industrial` ·
`5 Pasture` · `6 PermanentCrop` · `7 Residential` · `8 River` · `9 SeaLake`

### Preprocessing already applied by the dataset authors

Patches were cropped from Sentinel-2 scenes selected for low cloud cover and sampled across a
full year to maximise seasonal variance. Bands at 20 m and 60 m resolution were resampled to
10 m. All 27,000 patches were manually checked and images that were mislabelled, cloud-covered
or full of snow and ice were discarded.

Two consequences matter for modelling:

1. **No atmospheric correction was applied.** A visible colour cast remains in an appreciable
   number of patches. The authors deliberately kept these so that classifiers must cope with
   them, which makes colour a less reliable cue than it first appears (see Section 5.4).
2. **Patches within a class are deliberately heterogeneous** — different forest types, different
   industrial building types — which raises intra-class variance by design.

## 3. Experimental setup

Key settings: seed 42, 224 x 224 RGB inputs, batch size 32, 20 epochs, AdamW, cosine learning-rate schedule, cross-entropy loss, ImageNet-pretrained weights and full fine-tuning. Section 6.1 tabulates the full protocol.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import (
    EfficientNet_B0_Weights,
    ResNet50_Weights,
    efficientnet_b0,
    resnet50,
)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "EuroSAT_RGB"
SPLIT_CSV = PROJECT_ROOT / "data" / "splits" / "eurosat_split_seed42.csv"
RESNET_OUTPUT = PROJECT_ROOT / "outputs" / "notebook_resnet50_20epochs"
EFFICIENTNET_OUTPUT = PROJECT_ROOT / "outputs" / "notebook_efficientnet_b0_20epochs"
os.environ["TORCH_HOME"] = str(PROJECT_ROOT / "outputs" / "_torch_cache")
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / "outputs" / "_matplotlib_cache")

CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DATA_ROOT.is_dir(), f"Dataset not found: {DATA_ROOT}"
assert SPLIT_CSV.is_file(), f"Split CSV not found: {SPLIT_CSV}"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'CUDA GPU' if device.type == 'cuda' else device}")
print(f"Epochs: {EPOCHS}")

## 4. Data integrity and split verification

We verify the fixed split before training. Each file belongs to one partition, and a cross-split
hash check detects byte-identical images. Any duplicate across partitions would inflate the
reported results.

In [ ]:
# The fixed split is checked before either model sees the data.
split_df = pd.read_csv(SPLIT_CSV)
required_columns = {"filepath", "label", "class_id", "split"}
assert required_columns.issubset(split_df.columns)
assert len(split_df) == 27_000
assert split_df["filepath"].is_unique
assert set(split_df["split"]) == {"train", "val", "test"}

split_paths = {
    name: set(split_df.loc[split_df["split"] == name, "filepath"])
    for name in ("train", "val", "test")
}
cross_split_path_overlap = sum(
    len(split_paths[left] & split_paths[right])
    for left, right in (("train", "val"), ("train", "test"), ("val", "test"))
)
missing_files = sum(not (DATA_ROOT / path).is_file() for path in split_df["filepath"])

# Exact-file hash audit catches copied image files under different names/splits.
hash_to_splits = {}
for row in split_df.itertuples(index=False):
    digest = hashlib.sha256((DATA_ROOT / row.filepath).read_bytes()).hexdigest()
    hash_to_splits.setdefault(digest, set()).add(row.split)
cross_split_exact_duplicates = sum(len(splits) > 1 for splits in hash_to_splits.values())

split_counts = pd.crosstab(split_df["label"], split_df["split"]).reindex(CLASS_NAMES)
split_counts = split_counts[["train", "val", "test"]]

assert missing_files == 0
assert cross_split_path_overlap == 0
assert cross_split_exact_duplicates == 0

print("DATA PREPROCESSING CHECK: SUCCESS")
print(f"Images: {len(split_df):,}")
print(f"Missing files: {missing_files}")
print(f"Cross-split path overlap: {cross_split_path_overlap}")
print(f"Cross-split exact-file duplicates: {cross_split_exact_duplicates}")
display(split_counts)

In [ ]:
# Use the project team's actual Dataset/DataLoader and augmentation implementation.
from src.data.dataloader import create_dataloaders

data = create_dataloaders(
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    model_type="pretrained",
    num_workers=NUM_WORKERS,
    seed=SEED,
    data_root=DATA_ROOT,
    split_csv=SPLIT_CSV,
)

loaders = {
    "train": data.train_loader,
    "validation": data.val_loader,
    "test": data.test_loader,
}
sample_images, sample_targets = next(iter(loaders["train"]))
assert tuple(sample_images.shape[1:]) == (3, IMAGE_SIZE, IMAGE_SIZE)

print("DATALOADERS: SUCCESS")
print(f"Train / validation / test: {len(data.train_dataset)} / {len(data.val_dataset)} / {len(data.test_dataset)}")
print(f"Transformed batch: {tuple(sample_images.shape)}")

## 5. Exploratory data analysis

In [ ]:
# 5.1 Class distribution and split composition.
class_counts = split_df["label"].value_counts().reindex(CLASS_NAMES)
distribution = split_counts.copy()
distribution["total"] = class_counts
distribution["share %"] = (class_counts / len(split_df) * 100).round(2)
imbalance_ratio = class_counts.max() / class_counts.min()

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(10), class_counts.to_numpy(), color="#4C78A8")
axes[0].axhline(class_counts.mean(), color="#E45756", linestyle="--",
                label=f"mean = {class_counts.mean():.0f}")
axes[0].set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
axes[0].set(title="Class distribution over the full dataset", ylabel="Images")
axes[0].legend(); axes[0].grid(alpha=0.25, axis="y")

bottom = np.zeros(10)
for split_name, colour in zip(["train", "val", "test"], ["#4C78A8", "#F58518", "#54A24B"]):
    axes[1].bar(range(10), split_counts[split_name].to_numpy(), bottom=bottom,
                label=split_name, color=colour)
    bottom += split_counts[split_name].to_numpy()
axes[1].set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
axes[1].set(title="Stratified split composition per class", ylabel="Images")
axes[1].legend(); axes[1].grid(alpha=0.25, axis="y")
figure.tight_layout()
plt.show()

split_share = (split_counts.T / split_counts.sum(axis=1)).T
display(distribution)
print(f"Imbalance ratio (largest / smallest class): {imbalance_ratio:.2f}")
print(f"Largest  : {class_counts.idxmax()} ({class_counts.max():,} images)")
print(f"Smallest : {class_counts.idxmin()} ({class_counts.min():,} images)")
print(f"Per-class train share ranges {split_share['train'].min():.3f}-{split_share['train'].max():.3f} "
      f"(stratification is exact)")

In [ ]:
# 5.2 What the classes actually look like, at native resolution.
sample_rng = np.random.default_rng(SEED)
SAMPLES_PER_CLASS = 6

figure, axes = plt.subplots(10, SAMPLES_PER_CLASS, figsize=(9, 15.5))
for row, class_name in enumerate(CLASS_NAMES):
    pool = split_df.loc[split_df["label"] == class_name, "filepath"].to_numpy()
    for column, relative_path in enumerate(sample_rng.choice(pool, SAMPLES_PER_CLASS, replace=False)):
        axis = axes[row, column]
        axis.imshow(Image.open(DATA_ROOT / relative_path))
        axis.set_xticks([]); axis.set_yticks([])
        if column == 0:
            axis.set_ylabel(class_name, rotation=0, ha="right", va="center", fontsize=9)
figure.suptitle("EuroSAT samples at native 64 x 64 resolution", y=0.999)
figure.tight_layout()
plt.show()

In [ ]:
# 5.3 Image properties, and what resizing to 224 x 224 really costs.
property_audit = split_df.sample(600, random_state=SEED)
observed_sizes, observed_modes = set(), set()
for relative_path in property_audit["filepath"]:
    with Image.open(DATA_ROOT / relative_path) as image:
        observed_sizes.add(image.size)
        observed_modes.add(image.mode)

assert len(observed_sizes) == 1, f"Patches are not uniform: {observed_sizes}"
native_size = observed_sizes.pop()
upscale_factor = IMAGE_SIZE / native_size[0]
GSD_METRES = 10

print(f"Audited {len(property_audit)} images")
print(f"  native size    : {native_size[0]} x {native_size[1]} px (uniform)")
print(f"  colour mode(s) : {observed_modes}")
print(f"  ground coverage: {native_size[0] * GSD_METRES / 1000:.2f} x "
      f"{native_size[1] * GSD_METRES / 1000:.2f} km at {GSD_METRES} m/px")
print()
print(f"Model input size : {IMAGE_SIZE} x {IMAGE_SIZE} px")
print(f"  upscale factor : {upscale_factor:.2f}x linear, {upscale_factor ** 2:.1f}x pixel count")
print(f"  effective GSD  : {GSD_METRES / upscale_factor:.2f} m/px of *interpolated* detail")

In [ ]:
# 5.4 Is colour alone enough to separate the classes?
colour_rng = np.random.default_rng(0)
IMAGES_PER_CLASS = 150

class_colour_means = []
for class_name in CLASS_NAMES:
    pool = split_df.loc[
        (split_df["label"] == class_name) & (split_df["split"] == "train"), "filepath"
    ].to_numpy()
    picks = colour_rng.choice(pool, IMAGES_PER_CLASS, replace=False)
    per_image_mean = np.stack([
        np.asarray(Image.open(DATA_ROOT / p), dtype=np.float32).reshape(-1, 3).mean(axis=0)
        for p in picks
    ])
    class_colour_means.append(per_image_mean.mean(axis=0))

colour_frame = pd.DataFrame(class_colour_means, index=CLASS_NAMES, columns=["R", "G", "B"])
centroids = colour_frame.to_numpy()
colour_distance = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=-1)

closest_pairs = sorted(
    (colour_distance[i, j], CLASS_NAMES[i], CLASS_NAMES[j])
    for i in range(10) for j in range(i + 1, 10)
)
closest_frame = pd.DataFrame(
    [{"class pair": f"{a} / {b}", "mean-RGB distance": round(d, 1)} for d, a, b in closest_pairs[:5]]
)

display(colour_frame.round(1))
print("Five closest class pairs in mean-RGB space:")
display(closest_frame)

### 5.5 Implications for modelling

Class imbalance is mild. The largest class has 1.5x as many images as the smallest (3,000
AnnualCrop vs 2,000 Pasture), and every class retains at least 1,400 training images. The
stratified split preserves these proportions to three decimal places. We use unweighted
cross-entropy and also report macro-F1, which gives each class equal weight in evaluation.

Upsampling from 64 x 64 to 224 x 224 creates twelve times as many pixels without adding spatial
information. It is required only by the ImageNet-pretrained backbones; the 10 m ground
resolution does not change. Detail that might separate a highway from a field boundary is not
recovered by resizing.

The sample grid points to the agricultural classes as the difficult group. AnnualCrop,
PermanentCrop, HerbaceousVegetation and Pasture share similar green-to-brown field textures,
whereas Forest, Residential, Industrial and SeaLake are more visually distinct. Section 10.2
checks whether the errors follow this pattern.

Mean RGB is a poor guide to these confusions. HerbaceousVegetation / Highway and Pasture / River
are the closest pairs in mean-RGB space but are rarely confused. Most errors instead occur
between AnnualCrop / PermanentCrop and HerbaceousVegetation / PermanentCrop, even though these
pairs are farther apart in colour. The pattern implicates spatial texture and layout. It agrees
with Helber et al.'s result that deep CNNs outperform Bag-of-Visual-Words by roughly 28 accuracy
points on EuroSAT. Colour jitter remains appropriate because average colour is unreliable and
some images retain the atmospheric colour casts described in Section 2.

## 6. Models, training protocol and attribution

### 6.1 Shared training protocol

Every model below is trained through the same harness, split, transforms and hyperparameters.
Architecture is the main systematic difference. Each architecture is evaluated once with seed 42,
so small between-model differences should be interpreted cautiously.

| Setting | Value |
|---|---|
| Initialisation | ImageNet-1k pretrained, full fine-tuning (no frozen layers) |
| Classifier head | Final layer replaced, 1000 -> 10 outputs |
| Input | 224 x 224 RGB, ImageNet channel normalisation |
| Augmentation (train only) | RandomResizedCrop (scale 0.90-1.0, ratio 0.90-1.10), horizontal + vertical flip (p=0.5), rotation up to 90°, colour jitter (brightness/contrast 0.20, saturation 0.15, hue 0.03) |
| Optimiser | AdamW, learning rate 1e-4, weight decay 1e-4 |
| Schedule | Cosine annealing over 20 epochs to 1e-6 |
| Loss | Unweighted cross-entropy (justified in Section 5.5) |
| Batch size | 32 |
| Precision | Mixed precision (AMP) |
| Seed | 42, set immediately before each model is constructed |
| Model selection | Checkpoint with the best **validation** macro-F1 |
| Test usage | Once per predefined training run, on the validation-selected checkpoint only |

Augmentation applies to the training split only; validation and test images are deterministically resized and normalised. Both pipelines are defined in `src/data/transforms.py` and reach the models through the shared `create_dataloaders` factory (Section 3), so all three models use the same data split and transform definitions. Flips and rotations are safe here because satellite patches have no canonical orientation.

Checkpoint selection uses macro-F1, giving every class equal weight despite the mild imbalance.

### 6.2 Attribution

**External components:**

- ResNet50 and EfficientNet-B0 architectures and their torchvision ImageNet-1k weights.
- DeiT-Tiny/16 architecture and its `timm` ImageNet-1k weights (Touvron et al., Facebook AI).
- The EuroSAT dataset itself (Helber et al., 2019).
- The `src.data` package — `Dataset`, transforms, stratified split generation and the
  `create_dataloaders` factory — written by our group's data-preprocessing member and imported
  unmodified in Section 3 so that every model provably sees identical data.

**Work completed for this notebook:** the leakage and integrity audit (Section 4), all exploratory
analysis (Section 5), the training / evaluation harness below including the confusion-matrix
metric implementation, checkpoint selection, and all comparison and error analysis in
Sections 10-13. Only each model's final classification layer is modified, from 1000 ImageNet
classes to the 10 EuroSAT classes.

In [ ]:
def confusion_from_predictions(targets, predictions, num_classes):
    indices = targets.to(torch.int64) * num_classes + predictions.to(torch.int64)
    return torch.bincount(indices, minlength=num_classes**2).reshape(num_classes, num_classes)


def metrics_from_confusion(confusion):
    matrix = confusion.to(torch.float64)
    support = matrix.sum(dim=1)
    predicted = matrix.sum(dim=0)
    true_positive = matrix.diag()
    precision = torch.where(predicted > 0, true_positive / predicted, 0.0)
    recall = torch.where(support > 0, true_positive / support, 0.0)
    f1 = torch.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)
    return {
        "accuracy": float(true_positive.sum() / matrix.sum()),
        "macro_f1": float(f1.mean()),
        "precision_per_class": precision.tolist(),
        "recall_per_class": recall.tolist(),
        "f1_per_class": f1.tolist(),
        "confusion_matrix": confusion.tolist(),
    }


def run_epoch(model, loader, criterion, device, num_classes, optimizer=None, scaler=None, use_amp=False):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_examples = 0
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    started = time.perf_counter()

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, targets)  # CrossEntropy loss for this batch
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        predictions = logits.argmax(dim=1)
        batch_size = targets.size(0)
        total_loss += float(loss.detach()) * batch_size
        total_examples += batch_size
        confusion += confusion_from_predictions(
            targets.detach().cpu(), predictions.detach().cpu(), num_classes
        )

    metrics = metrics_from_confusion(confusion)
    metrics["loss"] = total_loss / total_examples  # mean epoch loss
    metrics["samples"] = total_examples
    metrics["seconds"] = time.perf_counter() - started
    return metrics


def save_training_curves(history, path):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(frame["epoch"], frame["train_loss"], label="Train")
    axes[0].plot(frame["epoch"], frame["val_loss"], label="Validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(frame["epoch"], frame["train_accuracy"], label="Train")
    axes[1].plot(frame["epoch"], frame["val_accuracy"], label="Validation")
    axes[1].set(title="Accuracy", xlabel="Epoch")
    axes[2].plot(frame["epoch"], frame["train_macro_f1"], label="Train")
    axes[2].plot(frame["epoch"], frame["val_macro_f1"], label="Validation")
    axes[2].set(title="Macro-F1", xlabel="Epoch")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def save_normalized_confusion(confusion_matrix, class_names, path):
    matrix = np.asarray(confusion_matrix, dtype=float)
    normalized = np.divide(
        matrix,
        matrix.sum(axis=1, keepdims=True),
        out=np.zeros_like(matrix),
        where=matrix.sum(axis=1, keepdims=True) != 0,
    )
    fig, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
    axis.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    axis.set_yticks(range(len(class_names)), class_names)
    axis.set(xlabel="Predicted", ylabel="True", title="Normalized confusion matrix")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def train_and_evaluate(model, loaders, device, model_name, output_dir, epochs=20):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / "best_model.pt"

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=LEARNING_RATE * 0.01
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history = []
    best_epoch = 0
    best_val_macro_f1 = -1.0
    training_started = time.perf_counter()

    print(f"{model_name}: starting real {epochs}-epoch training")
    for epoch in range(1, epochs + 1):
        learning_rate = optimizer.param_groups[0]["lr"]
        train_metrics = run_epoch(
            model, loaders["train"], criterion, device, len(CLASS_NAMES),
            optimizer=optimizer, scaler=scaler, use_amp=use_amp,
        )
        val_metrics = run_epoch(
            model, loaders["validation"], criterion, device, len(CLASS_NAMES),
            scaler=scaler, use_amp=use_amp,
        )

        row = {
            "epoch": epoch,
            "learning_rate": learning_rate,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "train_seconds": train_metrics["seconds"],
            "val_seconds": val_metrics["seconds"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)

        is_best = val_metrics["macro_f1"] > best_val_macro_f1
        if is_best:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)

        marker = "  <-- saved new best_model.pt" if is_best else ""
        print(
            f"Epoch {epoch:02d}/{epochs} | lr={learning_rate:.6g} | "
            f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
            f"macro-F1={train_metrics['macro_f1']:.4f} | "
            f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
            f"macro-F1={val_metrics['macro_f1']:.4f}{marker}",
            flush=True,
        )
        scheduler.step()

    assert len(history) == epochs
    calculated_best = max(history, key=lambda item: item["val_macro_f1"])
    assert calculated_best["epoch"] == best_epoch
    assert checkpoint_path.is_file()

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    test_metrics = run_epoch(
        model, loaders["test"], criterion, device, len(CLASS_NAMES),
        scaler=scaler, use_amp=use_amp,
    )
    training_seconds = time.perf_counter() - training_started

    summary = {
        "status": "training_complete",
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_val_macro_f1,
        "best_validation_loss": calculated_best["val_loss"],
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "training_seconds": training_seconds,
        "checkpoint": str(checkpoint_path.relative_to(PROJECT_ROOT)),
    }
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (output_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
    save_training_curves(history, output_dir / "training_curves.png")
    save_normalized_confusion(
        test_metrics["confusion_matrix"], CLASS_NAMES,
        output_dir / "confusion_matrix_normalized.png",
    )

    print(f"Training completed: {len(history)}/{epochs} epochs")
    print(f"Best model: epoch {best_epoch}, validation macro-F1={best_val_macro_f1:.4f}")
    print(
        f"Held-out test: loss={test_metrics['loss']:.4f}, "
        f"accuracy={test_metrics['accuracy']:.4f}, macro-F1={test_metrics['macro_f1']:.4f}"
    )
    return history, summary, test_metrics


print("Training code loaded: loss calculation, 20-epoch loop, best-checkpoint rule and test evaluation are active.")

## 7. ResNet50 — 20-epoch fine-tuning

ResNet50 provides the reference point because the EuroSAT authors used this residual CNN in their published benchmark.

In [ ]:
def create_resnet50(num_classes=10):
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

set_seed(SEED)
resnet_model = create_resnet50(len(CLASS_NAMES)).to(device)
print(f"ResNet50 parameters: {sum(parameter.numel() for parameter in resnet_model.parameters()):,}")

In [ ]:
resnet_history, resnet_summary, resnet_test_metrics = train_and_evaluate(
    model=resnet_model,
    loaders=loaders,
    device=device,
    model_name="ResNet50",
    output_dir=RESNET_OUTPUT,
    epochs=EPOCHS,
)

In [ ]:
assert resnet_summary["epochs_completed"] == 20

display(pd.DataFrame(resnet_history).round(4))
display(pd.Series(resnet_summary, name="ResNet50 result"))

figure, axes = plt.subplots(1, 2, figsize=(16, 6))

history_frame = pd.DataFrame(resnet_history)

# Loss curves
axes[0].plot(
    history_frame["epoch"],
    history_frame["train_loss"],
    label="Train loss"
)
axes[0].plot(
    history_frame["epoch"],
    history_frame["val_loss"],
    label="Validation loss"
)

axes[0].set(
    title="ResNet50 loss",
    xlabel="Epoch",
    ylabel="Loss"
)

axes[0].set_xlim(1, 20)
axes[0].set_ylim(0, 0.5)
axes[0].grid(alpha=0.25)
axes[0].legend()

# Normalized confusion matrix
matrix = np.asarray(
    resnet_test_metrics["confusion_matrix"],
    dtype=float
)

row_sums = matrix.sum(axis=1, keepdims=True)

# 避免某一类别没有样本时出现除以 0
matrix = np.divide(
    matrix,
    row_sums,
    out=np.zeros_like(matrix),
    where=row_sums != 0
)

image = axes[1].imshow(
    matrix,
    cmap="Blues",
    vmin=0,
    vmax=1
)

axes[1].set(
    title="ResNet50 normalized confusion matrix",
    xlabel="Predicted",
    ylabel="True"
)

axes[1].set_xticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

axes[1].set_yticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES
)

# 在每个格子中显示数值
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]

        axes[1].text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            fontweight="bold",
            color="white" if value >= 0.50 else "black"
        )

# 添加颜色条
figure.colorbar(
    image,
    ax=axes[1],
    fraction=0.046,
    pad=0.04,
    label="Proportion"
)

figure.tight_layout()
plt.show()

## 8. EfficientNet-B0 — 20-epoch fine-tuning

EfficientNet-B0 is a compound-scaled CNN with roughly one sixth of ResNet50's parameters. Its results measure the accuracy retained by a much smaller convolutional model.

In [ ]:
def create_efficientnet_b0(num_classes=10):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

set_seed(SEED)
efficientnet_model = create_efficientnet_b0(len(CLASS_NAMES)).to(device)
print(f"EfficientNet-B0 parameters: {sum(parameter.numel() for parameter in efficientnet_model.parameters()):,}")

In [ ]:
efficientnet_history, efficientnet_summary, efficientnet_test_metrics = train_and_evaluate(
    model=efficientnet_model,
    loaders=loaders,
    device=device,
    model_name="EfficientNet-B0",
    output_dir=EFFICIENTNET_OUTPUT,
    epochs=EPOCHS,
)

In [ ]:
assert efficientnet_summary["epochs_completed"] == 20

display(pd.DataFrame(efficientnet_history).round(4))
display(pd.Series(efficientnet_summary, name="EfficientNet-B0 result"))

figure, axes = plt.subplots(1, 2, figsize=(16, 6))

history_frame = pd.DataFrame(efficientnet_history)

# Loss curves
axes[0].plot(
    history_frame["epoch"],
    history_frame["train_loss"],
    label="Train loss"
)
axes[0].plot(
    history_frame["epoch"],
    history_frame["val_loss"],
    label="Validation loss"
)

axes[0].set(
    title="EfficientNet-B0 loss",
    xlabel="Epoch",
    ylabel="Loss"
)
axes[0].set_xlim(1, 20)
axes[0].set_ylim(0, 0.5)
axes[0].grid(alpha=0.25)
axes[0].legend()

# Normalized confusion matrix
matrix = np.asarray(
    efficientnet_test_metrics["confusion_matrix"],
    dtype=float
)

row_sums = matrix.sum(axis=1, keepdims=True)

matrix = np.divide(
    matrix,
    row_sums,
    out=np.zeros_like(matrix),
    where=row_sums != 0
)

image = axes[1].imshow(
    matrix,
    cmap="Blues",
    vmin=0,
    vmax=1
)

axes[1].set(
    title="EfficientNet-B0 normalized confusion matrix",
    xlabel="Predicted",
    ylabel="True"
)

axes[1].set_xticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

axes[1].set_yticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES
)

# 在每个格子中显示数值
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]

        axes[1].text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            fontweight="bold",
            color="white" if value >= 0.5 else "black"
        )

# 添加颜色条
figure.colorbar(
    image,
    ax=axes[1],
    fraction=0.046,
    pad=0.04,
    label="Proportion"
)

figure.tight_layout()
plt.show()

## 9. DeiT-Tiny/16 — 20-epoch fine-tuning

DeiT-Tiny/16 is a distillation-trained Vision Transformer. Its 5.5M parameters are close to EfficientNet-B0, providing a roughly size-matched comparison between attention and convolution.

In [ ]:
def create_deit_tiny(num_classes=10):
    try:
        import timm
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError(
            "DeiT-Tiny requires timm. Install it with: pip install timm"
        ) from error
    return timm.create_model(
        "deit_tiny_patch16_224",
        pretrained=True,
        num_classes=num_classes,
    )

DEIT_OUTPUT = PROJECT_ROOT / "outputs" / "notebook_deit_tiny_20epochs"

set_seed(SEED)
deit_model = create_deit_tiny(len(CLASS_NAMES)).to(device)
print(f"DeiT-Tiny parameters: {sum(parameter.numel() for parameter in deit_model.parameters()):,}")

In [ ]:
deit_history, deit_summary, deit_test_metrics = train_and_evaluate(
    model=deit_model,
    loaders=loaders,
    device=device,
    model_name="DeiT-Tiny/16",
    output_dir=DEIT_OUTPUT,
    epochs=EPOCHS,
)

In [ ]:
assert deit_summary["epochs_completed"] == 20

display(pd.DataFrame(deit_history).round(4))
display(pd.Series(deit_summary, name="DeiT-Tiny/16 result"))

figure, axes = plt.subplots(1, 2, figsize=(16, 6))

history_frame = pd.DataFrame(deit_history)

# Loss curves
axes[0].plot(
    history_frame["epoch"],
    history_frame["train_loss"],
    label="Train loss"
)
axes[0].plot(
    history_frame["epoch"],
    history_frame["val_loss"],
    label="Validation loss"
)

axes[0].set(
    title="DeiT-Tiny loss",
    xlabel="Epoch",
    ylabel="Loss"
)
axes[0].set_xlim(1, 20)
axes[0].set_ylim(0, 0.5)

axes[0].grid(alpha=0.25)
axes[0].legend()

# Normalized confusion matrix
matrix = np.asarray(
    deit_test_metrics["confusion_matrix"],
    dtype=float
)

row_sums = matrix.sum(axis=1, keepdims=True)

matrix = np.divide(
    matrix,
    row_sums,
    out=np.zeros_like(matrix),
    where=row_sums != 0
)

image = axes[1].imshow(
    matrix,
    cmap="Blues",
    vmin=0,
    vmax=1
)

axes[1].set(
    title="DeiT-Tiny normalized confusion matrix",
    xlabel="Predicted",
    ylabel="True"
)

axes[1].set_xticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

axes[1].set_yticks(
    range(len(CLASS_NAMES)),
    CLASS_NAMES
)

# 在每个格子中显示百分比
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]

        axes[1].text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            fontweight="bold",
            color="white" if value >= 0.5 else "black"
        )

# 添加颜色条
figure.colorbar(
    image,
    ax=axes[1],
    fraction=0.046,
    pad=0.04,
    label="Proportion"
)

figure.tight_layout()
plt.show()

## 10. Cross-model comparison

In [ ]:
# 10.1 Headline comparison across all three architectures.
MODEL_RESULTS = [
    ("ResNet50", resnet_model, resnet_summary, resnet_test_metrics),
    ("EfficientNet-B0", efficientnet_model, efficientnet_summary, efficientnet_test_metrics),
    ("DeiT-Tiny/16", deit_model, deit_summary, deit_test_metrics),
]

comparison_rows = []
for name, model, summary, test_metrics in MODEL_RESULTS:
    confusion = np.asarray(test_metrics["confusion_matrix"])
    comparison_rows.append({
        "parameters (M)": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
        "test accuracy %": round(test_metrics["accuracy"] * 100, 2),
        "test macro-F1 %": round(test_metrics["macro_f1"] * 100, 2),
        "test errors": int(confusion.sum() - np.trace(confusion)),
        "val macro-F1 %": round(summary["best_validation_macro_f1"] * 100, 2),
        "best epoch": summary["best_epoch"],
        "train time (s)": round(summary["training_seconds"]),
        "checkpoint (MB)": round(Path(summary["checkpoint"]).stat().st_size / 1e6, 1),
    })

comparison = pd.DataFrame(comparison_rows, index=[row[0] for row in MODEL_RESULTS])
display(comparison)

test_images = int(np.asarray(resnet_test_metrics["confusion_matrix"]).sum())
accuracy_spread = comparison["test accuracy %"].max() - comparison["test accuracy %"].min()
print(f"Test set: {test_images:,} images")
print(f"Accuracy spread across the three models: {accuracy_spread:.2f} percentage points "
      f"= {accuracy_spread / 100 * test_images:.0f} images")
print(f"Most accurate    : {comparison['test accuracy %'].idxmax()}")
print(f"Most parameter-efficient : {comparison['parameters (M)'].idxmin()} "
      f"({comparison['parameters (M)'].min()}M parameters)")
print(f"Fastest to train : {comparison['train time (s)'].idxmin()} "
      f"({comparison['train time (s)'].min()} s)")

In [ ]:
# 10.2 Final test accuracy comparison across the three models.

accuracy = comparison["test accuracy %"]
colours = ["#4C78A8", "#F58518", "#54A24B"]

figure, axis = plt.subplots(figsize=(9, 5.5))
bars = axis.bar(accuracy.index, accuracy.values, color=colours, width=0.6)

axis.set(
    title="Test Accuracy Comparison Across the Three Models",
    xlabel="Model",
    ylabel="Test accuracy (%)",
    ylim=(0, 100),
)
axis.grid(axis="y", alpha=0.25)

# Put the exact value inside each bar so small differences remain visible
# without using a truncated y-axis.
for bar, value in zip(bars, accuracy.values):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value - 2.0,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        color="white",
        fontsize=10,
        fontweight="bold",
    )

figure.tight_layout()
plt.show()


In [ ]:
# 10.3 Where do the errors actually live? Per-class F1 and pooled confusions.
per_class_f1 = pd.DataFrame(
    {name: np.asarray(m["f1_per_class"]) * 100 for name, _, _, m in MODEL_RESULTS},
    index=CLASS_NAMES,
)
per_class_f1["mean"] = per_class_f1.mean(axis=1)
per_class_f1 = per_class_f1.sort_values("mean").round(2)

pooled_errors = sum(np.asarray(m["confusion_matrix"], dtype=float) for _, _, _, m in MODEL_RESULTS)
np.fill_diagonal(pooled_errors, 0)

figure, axes = plt.subplots(1, 2, figsize=(14, 4.8))
positions = np.arange(10)
for offset, (name, _, _, metrics) in zip([-0.27, 0.0, 0.27], MODEL_RESULTS):
    axes[0].bar(positions + offset, np.asarray(metrics["f1_per_class"]) * 100, width=0.27, label=name)
axes[0].set_xticks(positions, CLASS_NAMES, rotation=45, ha="right")
axes[0].set(title="Per-class test F1", ylabel="F1 (%)", ylim=(95, 100))
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25, axis="y")

image = axes[1].imshow(pooled_errors, cmap="Reds")
axes[1].set_xticks(positions, CLASS_NAMES, rotation=45, ha="right")
axes[1].set_yticks(positions, CLASS_NAMES)
axes[1].set(title="Misclassifications pooled over all three models",
            xlabel="Predicted", ylabel="True")
figure.colorbar(image, ax=axes[1], fraction=0.046)
figure.tight_layout()
plt.show()

symmetric_pairs = sorted(
    ((pooled_errors[i, j] + pooled_errors[j, i], CLASS_NAMES[i], CLASS_NAMES[j])
     for i in range(10) for j in range(i + 1, 10)),
    reverse=True,
)
worst_pairs = pd.DataFrame(
    [{"class pair": f"{a} <-> {b}", "errors (3 models pooled)": int(count)}
     for count, a, b in symmetric_pairs[:6]]
)

AGRICULTURAL = {"AnnualCrop", "HerbaceousVegetation", "Pasture", "PermanentCrop"}
agricultural_errors = sum(
    count for count, a, b in symmetric_pairs if a in AGRICULTURAL and b in AGRICULTURAL
)

display(per_class_f1)
display(worst_pairs)
print(f"The 4 agricultural classes account for "
      f"{agricultural_errors:.0f} of {pooled_errors.sum():.0f} pooled errors "
      f"({agricultural_errors / pooled_errors.sum() * 100:.1f}%), "
      f"from 6 of the 45 possible class pairs.")

## 11. Comparison with published results

The published EuroSAT benchmark provides an external reference for our results.

**Protocol difference.** Helber et al. use an 80/20 train/test split with no
validation set (21,600 training images). We use a stratified 70/15/15 split (18,900 training
images) so that checkpoint selection never touches test data. Our training split contains
2,700 fewer images and our test set is smaller, so the comparison is indicative rather than
exact.

In [ ]:
# 11.1 Our results against the published EuroSAT benchmark (RGB).
published_results = pd.DataFrame([
    {"source": "Helber et al. 2019", "model": "ResNet-50 (fine-tuned)",
     "split": "80/20", "train images": 21_600, "accuracy %": 98.57},
    {"source": "Helber et al. 2019", "model": "GoogLeNet (fine-tuned)",
     "split": "80/20", "train images": 21_600, "accuracy %": 98.18},
    {"source": "Helber et al. 2019", "model": "ResNet-50 (from scratch)",
     "split": "80/20", "train images": 21_600, "accuracy %": 96.43},
    {"source": "Helber et al. 2019", "model": "GoogLeNet (from scratch)",
     "split": "80/20", "train images": 21_600, "accuracy %": 96.02},
    {"source": "Helber et al. 2019", "model": "Shallow CNN (2 layers)",
     "split": "80/20", "train images": 21_600, "accuracy %": 87.96},
    {"source": "Helber et al. 2019", "model": "BoVW + SVM (SIFT, k=500)",
     "split": "80/20", "train images": 21_600, "accuracy %": 70.05},
])
our_results = pd.DataFrame([
    {"source": "This project", "model": name, "split": "70/15/15", "train images": 18_900,
     "accuracy %": round(metrics["accuracy"] * 100, 2)}
    for name, _, _, metrics in MODEL_RESULTS
])

benchmark_table = pd.concat([published_results, our_results], ignore_index=True)
benchmark_table = benchmark_table.sort_values("accuracy %", ascending=False, ignore_index=True)
display(benchmark_table)

published_best = 98.57
our_best_name = comparison["test accuracy %"].idxmax()
our_best = comparison["test accuracy %"].max()
print(f"Best published fine-tuned result : {published_best:.2f}% (ResNet-50, 21,600 train images)")
print(f"Our best                         : {our_best:.2f}% ({our_best_name}, 18,900 train images)")
print(f"Difference                       : {our_best - published_best:+.2f} pp "
      f"using {21_600 - 18_900:,} fewer training images")
print(f"Pretrained vs from-scratch gap reported by Helber et al.: "
      f"{98.57 - 96.43:+.2f} pp")

## 12. Discussion

### 12.1 Final test performance and model size

All three architectures achieved test accuracy above 98% on the fixed EuroSAT split. The
recorded differences are small: the gap between the highest and lowest result is less than one
percentage point. The accuracy chart in Section 10 therefore shows broadly comparable
classification performance rather than a large separation between the models.

EfficientNet-B0 and DeiT-Tiny/16 use substantially fewer parameters and smaller checkpoints than
ResNet50 while retaining similar test performance. This makes the compact models attractive when
storage is limited. Deployment latency and memory consumption were not measured, so parameter
count alone should not be interpreted as evidence of faster inference.

Each architecture was evaluated once with seed 42. Small differences between these runs may
reflect run-to-run variation, so the results do not establish a stable statistical ranking among
the three models. Repeated runs would be required before claiming that one architecture is
consistently more accurate.

### 12.2 Errors in the agricultural classes

The per-class analysis shows a similar error pattern across the three architectures. PermanentCrop,
Pasture, HerbaceousVegetation and AnnualCrop are among the most difficult categories, and a
substantial share of the pooled errors occurs between these agricultural classes.

This agrees with the EuroSAT authors' observation that their best classifier sometimes confuses
agricultural land classes. Annual crops, permanent crops, pasture and herbaceous vegetation share
field textures that are distinguished partly by crop cycle and land management. Those properties
are difficult to infer from a single 640 m x 640 m RGB patch. Section 5.4 also indicates that the
most confused pairs are not simply the closest classes in average colour, suggesting that spatial
and textural information is important.

### 12.3 Limitations

The random split does not account for spatial autocorrelation. Section 4 rules out duplicated files
across splits, but EuroSAT patches come from a limited number of Sentinel-2 scenes. Geographically
adjacent patches may therefore appear in different partitions and share acquisition conditions or
land-management practices. A split by country or scene would better estimate generalisation to a
new region.

Only the RGB bands were used, although Sentinel-2 provides 13 spectral bands. Additional bands,
particularly near-infrared and red-edge information, may help distinguish vegetation and crop
types. The fixed 20-epoch budget may also affect the comparison because different architectures
can converge at different rates.

### 12.4 Future work

1. Repeat each model with multiple random seeds and report the mean and standard deviation.
2. Adapt the first convolution or patch embedding to 13 channels and evaluate the additional bands.
3. Apply Grad-CAM or attention rollout to investigate errors in the agricultural classes.
4. Re-partition the data by country or scene to test geographically disjoint generalisation.


## 13. Conclusion

ResNet50, EfficientNet-B0 and DeiT-Tiny/16 all achieved strong classification performance on the
fixed EuroSAT test split. The final test accuracies were close, showing that all three architectures
were effective for this land-cover classification task.

The compact EfficientNet-B0 and DeiT-Tiny/16 models achieved performance close to the larger
ResNet50 while using substantially fewer parameters and smaller checkpoints. Model selection
should therefore consider storage, training cost and the intended deployment environment in
addition to test accuracy.

Across the three models, the remaining errors were concentrated mainly in visually similar
agricultural classes. This shared pattern suggests that future improvements may depend on richer
spectral information, geographically disjoint evaluation and targeted analysis of crop-related
confusions. Because each architecture was evaluated with one random seed, repeated runs are also
needed before drawing conclusions from the small differences between models.

## References

1. Helber, P., Bischke, M., Dengel, A., and Borth, D. (2019). *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*. IEEE JSTARS, 12(7), 2217-2226.
2. He, K., Zhang, X., Ren, S., and Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR.
3. Tan, M., and Le, Q. V. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML.
4. Touvron, H. et al. (2021). *Training Data-Efficient Image Transformers & Distillation through Attention*. ICML.
